# 🏥 Health Insurance Claim Prediction - Exploratory Data Analysis

## Overview
This notebook provides a comprehensive exploratory data analysis (EDA) of the health insurance claims dataset. We'll analyze patterns, distributions, and relationships that influence claim approval decisions.

## Business Context
Understanding factors that lead to claim denials helps:
- Insurance companies optimize their approval processes
- Healthcare providers better prepare claim submissions
- Patients understand potential claim risks

---

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# Configure display settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print("📚 Libraries loaded successfully!")

## 1. Data Loading and Initial Inspection

In [ ]:
# Load the dataset
df = pd.read_csv('data/insurance_encoded.csv')

print(f"📊 Dataset shape: {df.shape}")
print(f"📁 Columns: {df.columns.tolist()}")
print(f"🧮 Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

In [ ]:
# Display first few rows
print("👀 First 5 rows:")
display(df.head())

print("\n📋 Data types:")
display(df.dtypes)

In [ ]:
# Basic statistics
print("📈 Descriptive Statistics:")
display(df.describe())

## 2. Data Quality Assessment

In [ ]:
# Check for missing values
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

data_quality = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percent
})

print("🔍 Missing Data Analysis:")
display(data_quality[data_quality['Missing Count'] > 0])

if data_quality['Missing Count'].sum() == 0:
    print("✅ No missing values found!")

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"🔄 Duplicate rows: {duplicates}")

if duplicates > 0:
    print("⚠️ Found duplicate rows - consider removing them")
else:
    print("✅ No duplicate rows found!")

## 3. Target Variable Analysis

In [ ]:
# Analyze target variable distribution
target_counts = df['claim_status'].value_counts()
target_percent = df['claim_status'].value_counts(normalize=True) * 100

print("🎯 Claim Status Distribution:")
status_summary = pd.DataFrame({
    'Status': ['Approved', 'Denied'],
    'Count': [target_counts[0], target_counts[1]], 
    'Percentage': [target_percent[0], target_percent[1]]
})
display(status_summary)

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
colors = ['#2E8B57', '#CD5C5C']
ax1.bar(['Approved (0)', 'Denied (1)'], target_counts, color=colors)
ax1.set_title('Claim Status Distribution (Count)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Count')
for i, v in enumerate(target_counts):
    ax1.text(i, v + 10, str(v), ha='center', va='bottom', fontweight='bold')

# Pie plot
ax2.pie(target_counts, labels=['Approved (0)', 'Denied (1)'], 
        colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Claim Status Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Check for class imbalance
imbalance_ratio = target_counts.max() / target_counts.min()
print(f"⚖️ Class imbalance ratio: {imbalance_ratio:.2f}")
if imbalance_ratio > 2:
    print("⚠️ Significant class imbalance detected - consider balancing techniques")
else:
    print("✅ Classes are reasonably balanced")

## 4. Feature Distribution Analysis

In [ ]:
# Analyze continuous features
continuous_features = ['age', 'bmi', 'children', 'charges']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(continuous_features):
    # Histogram with density curve
    axes[i].hist(df[feature], bins=30, alpha=0.7, color='skyblue', density=True)
    
    # Add kernel density estimate
    df[feature].plot.density(ax=axes[i], color='red', linewidth=2)
    
    axes[i].set_title(f'{feature.title()} Distribution', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(feature.title())
    axes[i].set_ylabel('Density')
    
    # Add statistics
    mean_val = df[feature].mean()
    median_val = df[feature].median()
    axes[i].axvline(mean_val, color='orange', linestyle='--', label=f'Mean: {mean_val:.1f}')
    axes[i].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.1f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Analyze categorical features
categorical_features = ['sex', 'smoker', 'region']
feature_labels = {
    'sex': {0: 'Female', 1: 'Male'},
    'smoker': {0: 'No', 1: 'Yes'},
    'region': {0: 'Region 0', 1: 'Region 1', 2: 'Region 2', 3: 'Region 3'}
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, feature in enumerate(categorical_features):
    counts = df[feature].value_counts().sort_index()
    
    # Create labels
    if feature in feature_labels:
        labels = [feature_labels[feature][x] for x in counts.index]
    else:
        labels = counts.index
    
    axes[i].bar(labels, counts.values, color=plt.cm.Set3(range(len(counts))))
    axes[i].set_title(f'{feature.title()} Distribution', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Count')
    
    # Add value labels on bars
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 10, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Feature Relationships and Correlations

In [ ]:
# Correlation matrix
correlation_matrix = df.corr()

# Create correlation heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdBu_r', 
            center=0, fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Highlight strongest correlations with target
target_corr = correlation_matrix['claim_status'].abs().sort_values(ascending=False)
print("🎯 Features most correlated with claim status:")
display(target_corr.drop('claim_status').head(10))

In [ ]:
# Pairplot for key relationships
key_features = ['age', 'bmi', 'charges', 'claim_status']
pair_data = df[key_features].copy()
pair_data['claim_status'] = pair_data['claim_status'].map({0: 'Approved', 1: 'Denied'})

plt.figure(figsize=(12, 10))
sns.pairplot(pair_data, hue='claim_status', diag_kind='hist', 
             palette={'Approved': '#2E8B57', 'Denied': '#CD5C5C'})
plt.suptitle('Key Feature Relationships by Claim Status', y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 6. Target-Based Feature Analysis

In [ ]:
# Analyze continuous features by claim status
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(continuous_features):
    # Box plot
    df_plot = df.copy()
    df_plot['Status'] = df_plot['claim_status'].map({0: 'Approved', 1: 'Denied'})
    
    sns.boxplot(data=df_plot, x='Status', y=feature, ax=axes[i],
                palette={'Approved': '#2E8B57', 'Denied': '#CD5C5C'})
    axes[i].set_title(f'{feature.title()} by Claim Status', fontsize=12, fontweight='bold')
    
    # Add statistical test results
    approved_vals = df[df['claim_status'] == 0][feature]
    denied_vals = df[df['claim_status'] == 1][feature]
    
    # Simple statistical comparison
    mean_diff = denied_vals.mean() - approved_vals.mean()
    axes[i].text(0.5, 0.95, f'Mean Diff: {mean_diff:.2f}', 
                transform=axes[i].transAxes, ha='center', va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Analyze categorical features by claim status
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, feature in enumerate(categorical_features):
    # Create crosstab
    crosstab = pd.crosstab(df[feature], df['claim_status'], normalize='index') * 100
    
    # Create labels
    if feature in feature_labels:
        index_labels = [feature_labels[feature][x] for x in crosstab.index]
    else:
        index_labels = crosstab.index
    
    # Stacked bar chart
    crosstab.plot(kind='bar', ax=axes[i], 
                  color=['#2E8B57', '#CD5C5C'],
                  width=0.7)
    
    axes[i].set_title(f'Claim Status by {feature.title()}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(feature.title())
    axes[i].set_ylabel('Percentage')
    axes[i].legend(['Approved', 'Denied'], loc='upper right')
    axes[i].set_xticklabels(index_labels, rotation=45)
    
    # Add percentage labels
    for j, (idx, row) in enumerate(crosstab.iterrows()):
        cumsum = 0
        for k, val in enumerate(row):
            axes[i].text(j, cumsum + val/2, f'{val:.1f}%', 
                        ha='center', va='center', fontweight='bold', color='white')
            cumsum += val

plt.tight_layout()
plt.show()

## 7. Outlier Detection and Analysis

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

print("🔍 Outlier Analysis:")
print("=" * 50)

for feature in continuous_features:
    outliers, lower, upper = detect_outliers_iqr(df, feature)
    outlier_count = len(outliers)
    outlier_percent = (outlier_count / len(df)) * 100
    
    print(f"\n{feature.title()}:")
    print(f"  Outliers: {outlier_count} ({outlier_percent:.1f}%)")
    print(f"  Valid range: [{lower:.2f}, {upper:.2f}]")
    
    if outlier_count > 0:
        print(f"  Outlier range: [{outliers[feature].min():.2f}, {outliers[feature].max():.2f}]")

In [ ]:
# Visualize outliers
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(continuous_features):
    # Box plot to show outliers
    box_plot = axes[i].boxplot(df[feature], patch_artist=True)
    box_plot['boxes'][0].set_facecolor('lightblue')
    
    axes[i].set_title(f'{feature.title()} - Outlier Detection', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(feature.title())
    
    # Add outlier count
    outliers, _, _ = detect_outliers_iqr(df, feature)
    axes[i].text(0.5, 0.95, f'Outliers: {len(outliers)}', 
                transform=axes[i].transAxes, ha='center', va='top',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

## 8. Healthcare-Specific Insights

In [ ]:
# Healthcare risk factor analysis
print("🏥 Healthcare Risk Factor Analysis:")
print("=" * 50)

# Age groups analysis
df['age_group'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], 
                         labels=['Young (18-30)', 'Middle (31-45)', 
                                'Senior (46-60)', 'Elder (60+)'])

age_risk = df.groupby('age_group')['claim_status'].agg(['count', 'sum', 'mean']).round(3)
age_risk.columns = ['Total Claims', 'Denied Claims', 'Denial Rate']
age_risk['Approval Rate'] = 1 - age_risk['Denial Rate']

print("\n📊 Risk by Age Group:")
display(age_risk)

# BMI categories analysis
def categorize_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['bmi_category'] = df['bmi'].apply(categorize_bmi)

bmi_risk = df.groupby('bmi_category')['claim_status'].agg(['count', 'sum', 'mean']).round(3)
bmi_risk.columns = ['Total Claims', 'Denied Claims', 'Denial Rate']
bmi_risk['Approval Rate'] = 1 - bmi_risk['Denial Rate']

print("\n⚖️ Risk by BMI Category:")
display(bmi_risk)

# High-cost claims analysis
high_cost_threshold = df['charges'].quantile(0.75)
df['high_cost'] = df['charges'] >= high_cost_threshold

cost_analysis = df.groupby('high_cost')['claim_status'].agg(['count', 'sum', 'mean']).round(3)
cost_analysis.index = ['Regular Cost', 'High Cost']
cost_analysis.columns = ['Total Claims', 'Denied Claims', 'Denial Rate']
cost_analysis['Approval Rate'] = 1 - cost_analysis['Denial Rate']

print(f"\n💰 Risk by Cost Level (High Cost > ${high_cost_threshold:,.2f}):")
display(cost_analysis)

In [ ]:
# Visualize healthcare risk factors
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Age group risk
age_risk['Denial Rate'].plot(kind='bar', ax=axes[0,0], color='coral')
axes[0,0].set_title('Claim Denial Rate by Age Group', fontsize=14, fontweight='bold')
axes[0,0].set_ylabel('Denial Rate')
axes[0,0].tick_params(axis='x', rotation=45)

# BMI category risk
bmi_risk['Denial Rate'].plot(kind='bar', ax=axes[0,1], color='lightcoral')
axes[0,1].set_title('Claim Denial Rate by BMI Category', fontsize=14, fontweight='bold')
axes[0,1].set_ylabel('Denial Rate')
axes[0,1].tick_params(axis='x', rotation=45)

# Smoker vs non-smoker
smoker_analysis = df.groupby('smoker')['claim_status'].mean()
smoker_labels = ['Non-Smoker', 'Smoker']
axes[1,0].bar(smoker_labels, smoker_analysis.values, color=['lightgreen', 'salmon'])
axes[1,0].set_title('Claim Denial Rate: Smoker vs Non-Smoker', fontsize=14, fontweight='bold')
axes[1,0].set_ylabel('Denial Rate')

# Cost level risk
cost_analysis['Denial Rate'].plot(kind='bar', ax=axes[1,1], color='gold')
axes[1,1].set_title('Claim Denial Rate by Cost Level', fontsize=14, fontweight='bold')
axes[1,1].set_ylabel('Denial Rate')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 9. Key Findings and Recommendations

### 🔍 **Key Findings:**

1. **Class Distribution**: 
   - Dataset appears balanced between approved and denied claims
   - No extreme class imbalance issues

2. **High-Risk Factors**:
   - Smoking status shows significant impact on claim denials
   - Higher medical charges correlate with increased denial rates
   - Age and BMI categories show varying risk patterns

3. **Data Quality**:
   - No missing values detected
   - Some outliers present in charges and BMI
   - Data appears clean and ready for modeling

### 📋 **Recommendations for Model Development:**

1. **Feature Engineering**:
   - Create age and BMI category features
   - Consider interaction terms (e.g., smoker × age)
   - Normalize/scale continuous features

2. **Model Considerations**:
   - Test multiple algorithms (Random Forest, XGBoost, Logistic Regression)
   - Use stratified cross-validation
   - Focus on recall for denied claims (minimize false negatives)

3. **Business Impact**:
   - Implement fairness checks across demographic groups
   - Consider regulatory compliance (HIPAA, ACA)
   - Plan for model explainability for healthcare stakeholders

---

**Next Steps**: Proceed to model development pipeline with these insights in mind.